In [96]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [79]:
def load_raw_data(raw_data_path:Path, year_lower_bound:int) -> pd.DataFrame:
    raw_dfs = []

    for csv in sorted(os.listdir(raw_data_path)):
        if csv == ".gitkeep":
            continue
        
        if int(csv.split(".")[0]) >= year_lower_bound:
            raw_dfs.append(pd.read_csv(raw_data_path / csv))

    return pd.concat(raw_dfs)

# Load in raw_data
raw_df = load_raw_data(raw_data_path = Path.cwd()/"data"/"raw", year_lower_bound = 2000)

In [80]:
# Look at 5 random rows
raw_df.sample(5)

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
700,2022-0404,Indian Wells Masters,Hard,128,M,20220307,252,144750,30.0,NaN,...,52.0,38.0,19.0,14.0,7.0,10.0,32.0,1393.0,82.0,781.0
753,2005-352,Paris Masters,Carpet,48,M,20051031,30,104068,12.0,NaN,...,42.0,38.0,22.0,14.0,2.0,5.0,18.0,1460.0,35.0,905.0
20,2014-339,Brisbane,Hard,28,A,20131229,21,103819,1.0,NaN,...,30.0,16.0,7.0,7.0,4.0,9.0,6.0,4205.0,61.0,774.0
243,2021-0321,Stuttgart,Grass,32,A,20210607,287,105023,NaN,NaN,...,47.0,34.0,14.0,11.0,1.0,3.0,67.0,1023.0,101.0,794.0
2937,2013-605,Tour Finals,Hard,8,F,20131104,513,104925,2.0,NaN,...,34.0,21.0,11.0,9.0,6.0,10.0,2.0,10610.0,8.0,3330.0


In [81]:
# Look at all columns -> `data_dictionary.txt` has column definitions
raw_df.columns

Index(['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level',
       'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry',
       'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age',
       'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand',
       'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round',
       'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
       'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt',
       'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced',
       'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points'],
      dtype='object')

In [82]:
# Filter out all Davis Cup matches -> only care about ATP, Slams, Olympics
raw_df = raw_df[~raw_df["tourney_name"].str.contains("Davis Cup")]

In [106]:
# Fix tourney_date dtype
raw_df["tourney_date"] = pd.to_datetime(raw_df["tourney_date"])

In [ ]:
# Check tourney level -> all look good
raw_df["tourney_level"].unique()

array(['A', 'M', 'G', 'F', 'O'], dtype=object)

In [117]:
raw_df["score"].str.split(" ")

0                  [7-5, 4-6, 7-5]
1                       [7-5, 7-5]
2                       [6-3, 6-1]
3                       [6-4, 6-4]
4               [0-6, 7-6(7), 6-1]
                   ...            
2811    [1-4, 4-2, 4-3(6), 4-3(5)]
2812    [2-4, 4-3(5), 4-3(4), 4-2]
2813    [4-3(3), 2-4, 4-1, 4-3(5)]
2814       [3-4(4), 4-2, 4-2, 4-1]
2815    [3-4(2), 4-3(7), 4-2, 4-2]
Name: score, Length: 67919, dtype: object

In [121]:
raw_df["winner_rank"]

0        11.0
1       211.0
2        48.0
3        45.0
4       167.0
        ...  
2811     41.0
2812     41.0
2813    128.0
2814    138.0
2815    128.0
Name: winner_rank, Length: 67919, dtype: float64

In [83]:
# Look at winner and loser hand unique values
raw_df["winner_hand"].unique(), raw_df["loser_hand"].unique()

(array(['R', 'L', 'U'], dtype=object),
 array(['L', 'R', 'U', 'A'], dtype=object))

In [84]:
# Check players who have U, A, and nan hand values
raw_df[raw_df["winner_hand"] == "U"]["winner_name"].unique() # Looks like generally unknown players
raw_df[raw_df["loser_hand"] == "A"]["loser_name"].unique() # Only two players -> map A to R (ambidextrous but assume typically use R hand)

array(['Luke Jensen', 'Rabie Chaki'], dtype=object)

In [85]:
raw_df["winner_ht"].unique(), raw_df["winner_ht"].dtype
raw_df[raw_df["winner_ht"].isna()]

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
188,2000-315,Newport,Grass,32,A,20000710,10,103162,NaN,NaN,...,26.0,18.0,16.0,9.0,4.0,8.0,169.0,219.0,123.0,311.0
452,2000-329,Tokyo,Hard,56,A,20001009,10,102808,NaN,Q,...,51.0,37.0,24.0,15.0,4.0,8.0,417.0,57.0,48.0,786.0
473,2000-329,Tokyo,Hard,56,A,20001009,31,102808,NaN,Q,...,40.0,23.0,8.0,9.0,7.0,11.0,417.0,57.0,97.0,431.0
600,2000-339,Adelaide,Hard,32,A,20000103,10,103162,NaN,WC,...,26.0,17.0,25.0,13.0,5.0,10.0,205.0,198.0,127.0,369.0
1658,2000-429,Stockholm,Hard,32,A,20001120,10,103158,NaN,Q,...,43.0,29.0,22.0,14.0,2.0,6.0,395.0,61.0,215.0,174.0
1663,2000-429,Stockholm,Hard,32,A,20001120,15,103751,NaN,WC,...,56.0,36.0,19.0,14.0,2.0,5.0,621.0,23.0,35.0,960.0
2741,2000-620,Brighton,Hard,32,A,20001120,11,104014,NaN,WC,...,43.0,28.0,11.0,10.0,7.0,9.0,424.0,55.0,92.0,433.0
2932,2001-890,Shanghai,Hard,32,A,20010917,10,103403,NaN,NaN,...,47.0,33.0,26.0,14.0,8.0,10.0,350.0,85.0,109.0,386.0
2943,2001-890,Shanghai,Hard,32,A,20010917,21,103403,NaN,NaN,...,51.0,34.0,11.0,14.0,5.0,13.0,350.0,85.0,506.0,41.0
240,2002-315,Newport,Grass,32,A,20020708,15,102902,NaN,WC,...,77.0,49.0,17.0,17.0,4.0,9.0,381.0,75.0,116.0,352.0


In [86]:
raw_df["loser_ht"].unique(), raw_df["loser_ht"].dtype
raw_df[raw_df["loser_ht"].isna()] # Players without enough matches to have height data

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,...,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points
97,2000-311,Queen's Club,Grass,56,A,20000612,5,102533,NaN,NaN,...,50.0,32.0,26.0,14.0,5.0,8.0,114.0,360.0,498.0,41.0
199,2000-315,Newport,Grass,32,A,20000710,21,101727,5.0,NaN,...,36.0,23.0,12.0,8.0,4.0,7.0,81.0,512.0,169.0,219.0
219,2000-316,Bastad,Clay,32,A,20000710,10,102104,NaN,NaN,...,21.0,9.0,11.0,7.0,2.0,8.0,122.0,312.0,566.0,28.0
286,2000-319,Kitzbuhel,Clay,48,A,20000724,15,103344,NaN,NaN,...,45.0,28.0,5.0,9.0,2.0,5.0,77.0,559.0,700.0,16.0
448,2000-329,Tokyo,Hard,56,A,20001009,6,102133,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,142.0,269.0,1351.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
450,2022-0424,Dallas,Hard,32,A,20220207,276,105577,NaN,Q,...,28.0,14.0,2.0,6.0,3.0,8.0,150.0,454.0,NaN,NaN
80,2024-0336,Hong Kong,Hard,32,A,20240101,273,207518,6.0,NaN,...,46.0,32.0,16.0,11.0,5.0,8.0,27.0,1470.0,NaN,NaN
1296,2024-7694,Lyon,Clay,32,A,20240520,277,106415,NaN,NaN,...,27.0,15.0,6.0,6.0,6.0,11.0,72.0,741.0,316.0,166.0
2216,2024-560,Us Open,Hard,128,G,20240826,129,126128,NaN,NaN,...,73.0,45.0,15.0,15.0,9.0,14.0,57.0,910.0,1848.0,1.0


In [87]:
# Check if earlier match data dont' have heights for players that eventually
# get their heights recorded
def check_ht_rows(names, raw_df:pd.DataFrame = raw_df):
    for name in names:
        unique_ht_instances = raw_df[(
            (raw_df["loser_name"] == "Dejan Petrovic") | 
            (raw_df["loser_name"] == "Dejan Petrovic")
            )]["loser_ht"].unique().tolist()

        if len(unique_ht_instances) > 1:
            print(f"{name} has both na and recorded height in data:")
            print(unique_ht_instances)
        
    print("No rows found with height discrepancies.")

# Check for names with NA values for loser_ht
names = raw_df[raw_df["loser_ht"].isna()]["loser_name"].unique().tolist()
check_ht_rows(names = names)

# Check for names with NA values for winner_ht
names = raw_df[raw_df["winner_ht"].isna()]["winner_name"].unique().tolist()
check_ht_rows(names = names)

# There does not appear to be rows where ht is nan for a player and later
# defined for the same player

No rows found with height discrepancies.
No rows found with height discrepancies.


In [ ]:
math.floor(2.8)

2

In [ ]:
# Check what ages look like
raw_df["winner_age"].dtype, raw_df["winner_age"].unique()
raw_df["loser_age"].dtype, raw_df["loser_age"].unique()

# Age is a float but can probably just be an int
# To convert to an int, we need to round down -> 
# a person isn't considered their next age until their birthday
raw_df["winner_age"] = np.floor(raw_df["winner_age"]).astype(int)
raw_df["loser_age"] = np.floor(raw_df["loser_age"]).astype(int)

In [105]:
raw_df.columns

Index(['tourney_id', 'tourney_name', 'surface', 'draw_size', 'tourney_level',
       'tourney_date', 'match_num', 'winner_id', 'winner_seed', 'winner_entry',
       'winner_name', 'winner_hand', 'winner_ht', 'winner_ioc', 'winner_age',
       'loser_id', 'loser_seed', 'loser_entry', 'loser_name', 'loser_hand',
       'loser_ht', 'loser_ioc', 'loser_age', 'score', 'best_of', 'round',
       'minutes', 'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
       'w_SvGms', 'w_bpSaved', 'w_bpFaced', 'l_ace', 'l_df', 'l_svpt',
       'l_1stIn', 'l_1stWon', 'l_2ndWon', 'l_SvGms', 'l_bpSaved', 'l_bpFaced',
       'winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points'],
      dtype='object')

In [ ]:
# Create unseeded indicator column & fill NAs with 0
df1["winner_unseeded"] = df1["winner_seed"].isna().astype(int)
df1["loser_unseeded"] = df1["loser_seed"].isna().astype(int)
df1["winner_seed"] = df1["winner_seed"].fillna(0)
df1["loser_seed"] = df1["loser_seed"].fillna(0)